In [82]:
import os 
from dotenv import load_dotenv
import json
from langchain_gigachat.chat_models import GigaChat
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import JsonOutputParser

load_dotenv()

# Настройка модели GigaChat с расширенными параметрами
llm = GigaChat(
    credentials=os.getenv('GIGA_KEY'),
    model='GigaChat-2', 
    verify_ssl_certs=False,
    temperature=0.2, # настройка креативности
    max_tokens=1000 # максимальная длина ответа
)

# Проверка подключения
# response = llm.invoke('Привет! Как дела?')
# print(response.content)

In [83]:
# Определяем JSON-схему вручную
json_schema = {
    "type": "object",
    "properties": {
        "total_persons": {"type": "integer", "description": "Общее количество человек"},
        "count_adults": {"type": "integer", "description": "Количество взрослых"},
        "count_children": {"type": "integer", "description": "Количество детей"},
        "start_date": {"type": "string", "description": "Дата заезда"},
        "nights": {"type": "integer", "description": "Количество ночей проживания"},
        "price_per_day": {"type": "number", "description": "Желаемая цена за сутки"},
        "remarks": {"type": "string", "description": "Особые пожелания"}
    },
    "required": ["total_persons", "count_adults", "count_children", "start_date", "nights", "price_per_day"]
}

# Создаем парсер для JSON
json_parser = JsonOutputParser(schema=json_schema)

In [84]:
# Собираем промпт с детализированными инструкциями для модели
system_prompt = SystemMessagePromptTemplate.from_template("""
Ты — эксперт по анализу текстов заявок на аренду жилья.

Твоя задача — извлечь из заявки данные:
- total_persons — общее количество проживающих;
- count_adults — количество взрослых;
- count_children — количество детей;
- start_date — дата заезда;
- nights — количество ночей;
- price_per_day — желаемая цена за сутки;
- remarks — особые пожелания.

ДЕТАЛИЗИРОВАННЫЕ ИНСТРУКЦИИ:
1. Внимательно прочитай текст заявки.
2. Найди все упоминания количества людей: взрослых, детей, семьи, пары, одного человека.
3. Используй правила:
   - "семья с одним ребенком" = 3 человека: 2 взрослых и 1 ребенок;
   - "семья с двумя детьми" = 4 человека: 2 взрослых и 2 ребенка;
   - "пара", "двое", "молодая семья" = 2 взрослых;
   - "один", "студент", "индивидуально" = 1 взрослый;
   - если сказано "3 взрослых и 2 детей", total_persons = 5.
4. Если количество указано числом, используй это число.
5. Если числа представлены в диапазоне, бери максимум.
6. Если даты представлены в диапазоне, бери самую раннюю дату заезда.
7. Если цена указана диапазоном, бери максимум.
8. Если поле не указано:
   - для чисел верни 0;
   - для текста верни пустую строку "".
9. Не используй null.
10. Не добавляй пояснения, комментарии, Markdown или текст вокруг JSON.
11. Первый символ ответа должен быть открывающейся фигурной скобкой.
12. Последний символ ответа должен быть закрывающейся фигурной скобкой.
""")


user_prompt = HumanMessagePromptTemplate.from_template("""
Проанализируй заявку на аренду жилья:

{text}

Верни только JSON-объект строго по инструкции ниже.

{format_instructions}
""")

chat_prompt = ChatPromptTemplate.from_messages([system_prompt, user_prompt])

In [85]:
import pandas as pd

# Загрузка данных из csv в датафрейм df
df = pd.read_csv('rental_09.csv', sep=';')

In [86]:
# Создание цепочки и вызов
chain = chat_prompt | llm | json_parser
for _, row in df.iterrows():
    text = row['text']
    response: dict = chain.invoke({'text': text, 'format_instructions': json_parser.get_format_instructions()})
    print(json.dumps(response, ensure_ascii=False, indent=2))

{
  "total_persons": 6,
  "count_adults": 3,
  "count_children": 3,
  "start_date": "30.07",
  "nights": 10,
  "price_per_day": "",
  "remarks": ""
}
{
  "total_persons": 3,
  "count_adults": 2,
  "count_children": 1,
  "start_date": "12.07",
  "nights": 6,
  "price_per_day": 0,
  "remarks": "со всеми удобствами недалеко от моря.Приход в час ночи."
}
{
  "total_persons": 2,
  "count_adults": 2,
  "count_children": 0,
  "start_date": "2023-09-03",
  "nights": 7,
  "price_per_day": 0,
  "remarks": ""
}
{
  "total_persons": 4,
  "count_adults": 4,
  "count_children": 0,
  "start_date": "31.08.18",
  "nights": 7,
  "price_per_day": "",
  "remarks": ""
}
{
  "total_persons": 3,
  "count_adults": 3,
  "count_children": 0,
  "start_date": "13.06",
  "nights": 8,
  "price_per_day": "",
  "remarks": "желательно с удобствами в номере"
}
{
  "total_persons": 2,
  "count_adults": 2,
  "count_children": 0,
  "start_date": "7-8-2023",
  "nights": 7,
  "price_per_day": 700,
  "remarks": ""
}
{
  "tot

In [87]:
# correct = 0
# for _, row in df.iterrows():
#     if pd.to_numeric(row['result'], errors='coerce') == row['amount']:
#         correct += 1

# total = len(df)
# accuracy = correct / total

# print(f'Ошибок: {total - correct}')
# print(f'Точность: {accuracy:.1%}')